In [ ]:
# Σύνδεση με το Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# -*- coding: utf-8 -*-

# =========================
# 02 — PDF DOWNLOADER WITH VALIDATION
# =========================

from google.colab import drive
drive.mount('/content/drive')

import os
import time
import pandas as pd
import requests

# =========================
# ΡΥΘΜΙΣΕΙΣ
# =========================

CSV_PATH = "/content/drive/MyDrive/ΔΙΔΑΚΤΟΡΙΚΟ PhD/4o PAPER/clean_dataset.csv"

# Καλύτερα νέο folder για καθαρό validated run
OUTPUT_DIR = "/content/drive/MyDrive/ΔΙΔΑΚΤΟΡΙΚΟ PhD/4o PAPER/oa_pdf"
LOG_PATH = "/content/drive/MyDrive/ΔΙΔΑΚΤΟΡΙΚΟ PhD/4o PAPER/oa_pdf_download_log_validated.csv"

REPORT_DIR = "/content/drive/MyDrive/ΔΙΔΑΚΤΟΡΙΚΟ PhD/4o PAPER/pdf_download_reports_validated"

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(REPORT_DIR, exist_ok=True)

# =========================
# ΦΟΡΤΩΣΗ DATASET
# =========================

df = pd.read_csv(CSV_PATH)

print("Total papers:", len(df))

print("\nCitation bin counts:")
print(df["citation_bin"].value_counts(dropna=False))

print("\nOpen Access counts:")
print(df["is_oa"].value_counts(dropna=False))

# Κρατάμε μόνο low/high papers που είναι Open Access και έχουν διαθέσιμο pdf_url
oa_df = df[
    (df["citation_bin"].isin(["low", "high"])) &
    (df["is_oa"].astype(str).str.lower() == "true") &
    (df["pdf_url"].notna()) &
    (df["pdf_url"].astype(str).str.strip() != "")
].copy()

print("\nLow/high OA papers with pdf_url:", len(oa_df))
print(oa_df["citation_bin"].value_counts(dropna=False))

print("\nJournals in download set:")
print(oa_df["journal_short"].value_counts(dropna=False))


# =========================
# PDF VALIDATION FUNCTIONS
# =========================

def is_valid_pdf_file(path):
    """
    Ελέγχει αν ένα αρχείο είναι πραγματικό και σχετικά ολοκληρωμένο PDF.
    Δεν αρκεί να έχει κατάληξη .pdf.
    """
    try:
        if not os.path.exists(path):
            return False

        if os.path.getsize(path) < 1024:
            return False

        with open(path, "rb") as f:
            header = f.read(5)

            try:
                f.seek(-2048, os.SEEK_END)
            except OSError:
                f.seek(0)

            tail = f.read()

        return header == b"%PDF-" and b"%%EOF" in tail

    except Exception:
        return False


def normalize_pdf_url_for_download(url):
    """
    Προαιρετική κανονικοποίηση URLs.
    Δεν πειράζει τα ήδη direct PDF links.
    """
    url = str(url).strip()

    # Αν κάποτε έχεις MDPI article page χωρίς /pdf, το μετατρέπει σε PDF URL.
    # Αν έχει ήδη /pdf?version=..., δεν το αλλάζει.
    if "mdpi.com/" in url and "/pdf" not in url:
        url = url.rstrip("/") + "/pdf"

    return url


# =========================
# DOWNLOAD FUNCTION
# =========================

def download_pdf(url, save_path, timeout=60):
    """
    Κατεβάζει PDF με ασφαλή τρόπο:
    - γράφει πρώτα σε .part αρχείο
    - ελέγχει status code
    - ελέγχει ότι αρχίζει με %PDF
    - ελέγχει Content-Length όταν υπάρχει
    - κάνει τελικό PDF validation
    - μόνο μετά το μετακινεί στο κανονικό .pdf path
    """

    headers = {
        "User-Agent": (
            "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
            "AppleWebKit/537.36 (KHTML, like Gecko) "
            "Chrome/120.0.0.0 Safari/537.36"
        ),
        "Accept": "application/pdf,application/octet-stream;q=0.9,*/*;q=0.8",
        "Accept-Language": "en-US,en;q=0.9",
        "Connection": "keep-alive",
    }

    temp_path = save_path + ".part"

    # Καθαρίζουμε τυχόν παλιό .part
    if os.path.exists(temp_path):
        try:
            os.remove(temp_path)
        except Exception:
            pass

    try:
        response = requests.get(
            url,
            headers=headers,
            stream=True,
            timeout=timeout,
            allow_redirects=True
        )

        status_code = response.status_code
        content_type = response.headers.get("Content-Type", "").lower()
        content_length = response.headers.get("Content-Length")
        final_url = response.url

        if status_code != 200:
            return False, f"HTTP {status_code} | Content-Type: {content_type} | Final URL: {final_url}"

        iterator = response.iter_content(chunk_size=8192)

        first_chunk = next(iterator, b"")

        if not first_chunk:
            return False, f"Empty response | Content-Type: {content_type} | Final URL: {final_url}"

        if not first_chunk.startswith(b"%PDF"):
            sample = first_chunk[:300].decode("utf-8", errors="ignore").replace("\n", " ").strip()
            return False, f"Not a PDF | Content-Type: {content_type} | Final URL: {final_url} | Sample: {sample[:200]}"

        bytes_written = 0

        with open(temp_path, "wb") as f:
            f.write(first_chunk)
            bytes_written += len(first_chunk)

            for chunk in iterator:
                if chunk:
                    f.write(chunk)
                    bytes_written += len(chunk)

        # Έλεγχος Content-Length, όταν υπάρχει
        if content_length is not None:
            try:
                expected = int(content_length)
                if bytes_written < expected:
                    if os.path.exists(temp_path):
                        os.remove(temp_path)
                    return False, f"Incomplete download | wrote {bytes_written} of {expected} bytes"
            except ValueError:
                pass

        # Τελικό validation
        if not is_valid_pdf_file(temp_path):
            if os.path.exists(temp_path):
                os.remove(temp_path)
            return False, "Downloaded file failed final PDF validation"

        # Μόνο αν είναι έγκυρο, το μετακινούμε στο τελικό path
        os.replace(temp_path, save_path)

        return True, f"Downloaded and validated | bytes: {bytes_written}"

    except Exception as e:
        if os.path.exists(temp_path):
            try:
                os.remove(temp_path)
            except Exception:
                pass

        return False, str(e)


# =========================
# DOWNLOAD LOOP
# =========================

logs = []

for idx, row in oa_df.iterrows():

    url = normalize_pdf_url_for_download(row["pdf_url"])

    # Αν υπάρχει στήλη filename, τη χρησιμοποιούμε.
    if "filename" in row and pd.notna(row["filename"]):
        filename = str(row["filename"]).strip()
        if not filename.lower().endswith(".pdf"):
            filename += ".pdf"
    else:
        filename = f"{idx}.pdf"

    save_path = os.path.join(OUTPUT_DIR, filename)

    # Αν υπάρχει ήδη, ελέγχουμε αν είναι όντως valid PDF
    if os.path.exists(save_path):
        if is_valid_pdf_file(save_path):
            logs.append({
                "index": idx,
                "filename": filename,
                "journal": row.get("journal", None),
                "journal_short": row.get("journal_short", None),
                "year": row.get("year", None),
                "citation_bin": row.get("citation_bin", None),
                "pdf_url": url,
                "success": True,
                "message": "Already exists and valid PDF",
                "file_size_bytes": os.path.getsize(save_path)
            })
            continue
        else:
            print(f"Existing invalid PDF removed: {filename}")
            try:
                os.remove(save_path)
            except Exception:
                pass

    print(f"Downloading {filename} | {row.get('journal_short', '')} | {row.get('citation_bin', '')}")

    success, message = download_pdf(url, save_path)

    file_size = os.path.getsize(save_path) if os.path.exists(save_path) else 0

    logs.append({
        "index": idx,
        "filename": filename,
        "journal": row.get("journal", None),
        "journal_short": row.get("journal_short", None),
        "year": row.get("year", None),
        "citation_bin": row.get("citation_bin", None),
        "pdf_url": url,
        "success": success,
        "message": message,
        "file_size_bytes": file_size
    })

    # Μικρή καθυστέρηση για να μη γίνονται πολύ επιθετικά requests
    if "mdpi.com" in url:
        time.sleep(3)
    elif "frontiersin.org" in url:
        time.sleep(1.5)
    else:
        time.sleep(1)


# =========================
# ΑΠΟΘΗΚΕΥΣΗ LOG
# =========================

log_df = pd.DataFrame(logs)
log_df["success"] = log_df["success"].astype(bool)

log_df.to_csv(LOG_PATH, index=False, encoding="utf-8-sig")

print("\nDone.")
print("Successful validated downloads:", int(log_df["success"].sum()))
print("Failed downloads:", int((~log_df["success"]).sum()))
print("PDFs saved in:", OUTPUT_DIR)
print("Log saved in:", LOG_PATH)


# =========================
# EXTRA VALIDATION OF OUTPUT FOLDER
# =========================

print("\n==============================")
print("0. FINAL FOLDER VALIDATION")
print("==============================")

pdf_files = [
    f for f in os.listdir(OUTPUT_DIR)
    if f.lower().endswith(".pdf")
]

valid_files = []
invalid_files = []

for f in pdf_files:
    path = os.path.join(OUTPUT_DIR, f)

    if is_valid_pdf_file(path):
        valid_files.append(f)
    else:
        invalid_files.append(f)

print("Total .pdf files in folder:", len(pdf_files))
print("Valid PDF files:", len(valid_files))
print("Invalid PDF files:", len(invalid_files))

if invalid_files:
    print("\nSample invalid files:")
    print(invalid_files[:20])


# =========================
# REPORTS: PDF AVAILABILITY / DOWNLOAD COVERAGE
# =========================

print("\n==============================")
print("1. VALID PDF DISTRIBUTION")
print("==============================")

valid_pdf_df = log_df[log_df["success"] == True].copy()

print("\nValid downloaded PDFs:", len(valid_pdf_df))

print("\nValid PDFs by citation_bin:")
print(valid_pdf_df["citation_bin"].value_counts(dropna=False))

print("\nValid PDFs by citation_bin (%):")
print((valid_pdf_df["citation_bin"].value_counts(normalize=True, dropna=False) * 100).round(2))

valid_bin_report = (
    valid_pdf_df["citation_bin"]
    .value_counts(dropna=False)
    .rename_axis("citation_bin")
    .reset_index(name="valid_pdf_count")
)

if len(valid_bin_report) > 0:
    valid_bin_report["valid_pdf_percent"] = (
        valid_bin_report["valid_pdf_count"] /
        valid_bin_report["valid_pdf_count"].sum() * 100
    ).round(2)

valid_bin_report.to_csv(
    os.path.join(REPORT_DIR, "valid_pdfs_by_citation_bin.csv"),
    index=False,
    encoding="utf-8-sig"
)


print("\n==============================")
print("2. DOWNLOAD SUCCESS BY JOURNAL")
print("==============================")

journal_success_report = (
    log_df
    .groupby(["journal", "journal_short"], dropna=False)
    .agg(
        attempted_pdfs=("filename", "count"),
        successful_downloads=("success", "sum")
    )
    .reset_index()
)

journal_success_report["failed_downloads"] = (
    journal_success_report["attempted_pdfs"] -
    journal_success_report["successful_downloads"]
)

journal_success_report["success_rate"] = (
    journal_success_report["successful_downloads"] /
    journal_success_report["attempted_pdfs"]
).round(4)

journal_success_report["success_rate_percent"] = (
    journal_success_report["success_rate"] * 100
).round(2)

journal_success_report = journal_success_report.sort_values(
    "success_rate",
    ascending=False
)

print(journal_success_report)

journal_success_report.to_csv(
    os.path.join(REPORT_DIR, "download_success_by_journal.csv"),
    index=False,
    encoding="utf-8-sig"
)


print("\n==============================")
print("3. DOWNLOAD SUCCESS BY JOURNAL AND CITATION BIN")
print("==============================")

journal_bin_report = (
    log_df
    .groupby(["journal", "journal_short", "citation_bin"], dropna=False)
    .agg(
        attempted_pdfs=("filename", "count"),
        successful_downloads=("success", "sum")
    )
    .reset_index()
)

journal_bin_report["failed_downloads"] = (
    journal_bin_report["attempted_pdfs"] -
    journal_bin_report["successful_downloads"]
)

journal_bin_report["success_rate"] = (
    journal_bin_report["successful_downloads"] /
    journal_bin_report["attempted_pdfs"]
).round(4)

journal_bin_report["success_rate_percent"] = (
    journal_bin_report["success_rate"] * 100
).round(2)

journal_bin_report = journal_bin_report.sort_values(
    ["journal", "citation_bin"]
)

print(journal_bin_report)

journal_bin_report.to_csv(
    os.path.join(REPORT_DIR, "download_success_by_journal_and_citation_bin.csv"),
    index=False,
    encoding="utf-8-sig"
)


print("\n==============================")
print("4. VALID PDF CROSS-TAB: JOURNAL x CITATION_BIN")
print("==============================")

valid_journal_bin_crosstab = pd.crosstab(
    valid_pdf_df["journal_short"],
    valid_pdf_df["citation_bin"],
    margins=True
)

print(valid_journal_bin_crosstab)

valid_journal_bin_crosstab.to_csv(
    os.path.join(REPORT_DIR, "valid_pdf_journal_x_citation_bin.csv"),
    encoding="utf-8-sig"
)


print("\n==============================")
print("5. FAILURE REASONS")
print("==============================")

failure_report = (
    log_df[log_df["success"] == False]
    .groupby("message", dropna=False)
    .agg(failed_count=("filename", "count"))
    .reset_index()
    .sort_values("failed_count", ascending=False)
)

print(failure_report.head(30))

failure_report.to_csv(
    os.path.join(REPORT_DIR, "download_failure_reasons.csv"),
    index=False,
    encoding="utf-8-sig"
)


print("\n==============================")
print("6. FINAL SUMMARY")
print("==============================")

total_attempted = len(log_df)
total_success = int(log_df["success"].sum())
total_failed = total_attempted - total_success

print(f"Total attempted PDFs: {total_attempted}")
print(f"Successful validated downloads: {total_success}")
print(f"Failed downloads: {total_failed}")

if total_attempted > 0:
    print(f"Overall success rate: {total_success / total_attempted * 100:.2f}%")
else:
    print("Overall success rate: N/A")

print("\nReports saved in:")
print(REPORT_DIR)

END.